목적: "모델에 넣을 입력 데이터 만들기"

In [5]:
import pandas as pd
import numpy as np

# 데이터 불러오기
df = pd.read_csv('../data/processed/data.csv', parse_dates=['date'])

In [6]:
def get_season(date):
    """
    1 = 봄 (3,4,5월)
    2 = 여름 (6,7,8월)
    3 = 가을 (9,10,11월)
    4 = 겨울 (12,1,2월)
    """
    month = date.month
    if month in [3, 4, 5]:
        return 1
    elif month in [6, 7, 8]:
        return 2
    elif month in [9, 10, 11]:
        return 3
    else:
        return 4

df['season'] = df['date'].apply(get_season)

In [7]:
# 참가자를 4명씩 그룹으로 묶기
unique_participants = df['participant_id'].unique()
np.random.seed(42)
np.random.shuffle(unique_participants)

group_size = 4
group_mapping = {}
for i, pid in enumerate(unique_participants):
    group_id = i // group_size + 1
    group_mapping[pid] = group_id

df['group_id'] = df['participant_id'].map(group_mapping)

In [8]:
# 걸음수, 칼로리, 운동시간, 거리의 14일 이동 평균
rolling_cols = ['daily_steps', 'calories_burned', 'duration_minutes']

for col in rolling_cols:
    df[f'{col}_14d_avg'] = df.groupby('participant_id')[col].transform(
        lambda x: x.rolling(window=14, min_periods=7).mean()
    )

# 거리 계산 (1000보 ≈ 0.8km)
df['distance_km'] = df['daily_steps'] * 0.8 / 1000
df['distance_km_14d_avg'] = df.groupby('participant_id')['distance_km'].transform(
    lambda x: x.rolling(window=14, min_periods=7).mean()
)

In [9]:
# 날짜별 그룹 평균
group_stats = df.groupby(['group_id', 'date']).agg({
    'participant_id': 'count',           # 그룹 인원
    'daily_steps_14d_avg': 'mean',       # 그룹 평균 걸음수
    'calories_burned_14d_avg': 'mean',
    'duration_minutes_14d_avg': 'mean',
    'distance_km_14d_avg': 'mean',
    'age': 'mean',                       # 평균 나이
    'bmi': 'mean',                       # 평균 BMI
    'resting_heart_rate': 'mean',        # 평균 심박수
    'season': 'first'                    # 계절
}).reset_index()

In [10]:
# 목표 = 현재 평균 * 인원수 * 1.1
group_stats['goal_steps'] = (
    group_stats['group_avg_steps_14'] * 
    group_stats['member_count'] * 
    1.1
).round()

# 실제 달성값 = 현재 평균 * 인원수
group_stats['actual_steps'] = (
    group_stats['group_avg_steps_14'] * 
    group_stats['member_count']
).round()

# 달성률 = 실제 / 목표 (노이즈 추가)
noise_factor = np.random.uniform(0.85, 1.15, len(group_stats))
group_stats['achievement_rate_steps'] = (
    group_stats['actual_steps'] / 
    group_stats['goal_steps'] * 
    noise_factor
).clip(0.3, 1.5)

KeyError: 'group_avg_steps_14'

In [ ]:
def calculate_next_goal(current_avg, achievement_rate, member_count):
    """달성률에 따른 다음 목표 계산"""
    if achievement_rate >= 0.9:
        growth_rate = 1.12  # 12% 증가
    elif achievement_rate >= 0.7:
        growth_rate = 1.08  # 8% 증가
    elif achievement_rate >= 0.5:
        growth_rate = 1.03  # 3% 증가
    else:
        growth_rate = 0.98  # 2% 감소
    
    return current_avg * member_count * growth_rate

# 다음 목표 계산
group_stats['next_goal_steps'] = group_stats.apply(
    lambda x: calculate_next_goal(
        x['group_avg_steps_14'], 
        x['achievement_rate_steps'], 
        x['member_count']
    ), 
    axis=1
).round()

In [ ]:
steps_features = [
    'member_count',
    'group_avg_steps_14',
    'achievement_rate_steps',
    'avg_age',
    'avg_bmi',
    'avg_resting_hr',
    'season',
    'user_min_steps',
    'next_goal_steps'  # 타겟
]

training_steps = group_stats_clean[steps_features].copy()
training_steps.columns = [
    'member_count',
    'group_avg_steps_14',
    'last_week_achievement_rate',  # 컬럼명 변경
    'avg_age',
    'avg_bmi',
    'avg_resting_hr',
    'season',
    'user_min_steps',
    'next_goal_steps'
]

training_steps.to_csv('../data/processed/training_steps.csv', index=False)

In [ ]:
kcal_features = [
    'member_count',
    'group_avg_kcal_14',
    'achievement_rate_kcal',
    'avg_age',
    'avg_bmi',
    'avg_resting_hr',
    'season',
    'user_min_kcal',
    'next_goal_kcal'  # 타겟
]

training_kcal = group_stats_clean[kcal_features].copy()
training_kcal.columns = [
    'member_count',
    'group_avg_kcal_14',
    'last_week_achievement_rate',  # 컬럼명 변경
    'avg_age',
    'avg_bmi',
    'avg_resting_hr',
    'season',
    'user_min_kcal',
    'next_goal_kcal'
]

training_steps.to_csv('../data/processed/training_kcal.csv', index=False)